In [110]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

## Wczytywanie danych

In [111]:
data_path = Path('ml-latest-small')

In [112]:
movies_df = pd.read_csv(data_path / 'movies.csv')
tags_df = pd.read_csv(data_path / 'tags.csv')
ratings_df = pd.read_csv(data_path / 'ratings.csv')

In [113]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [114]:
tags_df

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


In [115]:
ratings_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


## Generowanie Cech Filmów

In [116]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [ ]:
movies_df['year'] = pd.to_numeric(
    movies_df['title'].str.extract(r'\((\d{4})\)\s*$')[0],
    errors='coerce'
)

In [ ]:
genres_expanded = movies_df['genres'].str.get_dummies(sep='|')

In [ ]:
genres_expanded

In [ ]:
genres_expanded.sum()

In [ ]:
movies_df = pd.concat([movies_df, genres_expanded], axis=1).drop(columns=['genres'])

In [ ]:
movies_df

## Cechy Uzytkownikow

In [ ]:
tags_df

In [ ]:
watched_movies_count = ratings_df.groupby('userId')['movieId'].nunique()
ratings_stats = ratings_df.groupby('userId')['rating'].agg(['max', 'min', 'mean'])
tags_count = tags_df.groupby('userId')['tag'].count()

In [ ]:
ratings_df['timestamp'] = pd.to_datetime(ratings_df['timestamp'], unit='s')
time_range= ratings_df.groupby('userId')['timestamp'].agg(['max', 'min'])
time_range['time_range'] = (time_range['max'] - time_range['min']).dt.total_seconds()
movies_per_time_range = watched_movies_count / time_range['time_range']

In [ ]:
tags_df['tag_length'] = tags_df['tag'].str.len()

In [ ]:
tags_length_stats = tags_df.groupby('userId')['tag_length'].agg(['max', 'min', 'mean'])

In [ ]:
users_df = pd.DataFrame({
    'number_of_watched_movies': watched_movies_count,
    'max_rating': ratings_stats ['max'],
    'min_rating': ratings_stats ['min'],
    'mean_rating': ratings_stats ['mean'],
    'number_of_tags': tags_count,
    'movies_per_time_range': movies_per_time_range,
    'max_tag_length': tags_length_stats['max'],
    'min_tag_length': tags_length_stats['min'],
    'mean_tag_length': tags_length_stats['mean']
})

In [ ]:
users_df.fillna(0, inplace=True)

In [ ]:
users_df.reset_index(inplace=True)

## Podział na zbiór treningowy i testowy

In [ ]:
user_train, user_test = train_test_split(users_df, test_size=0.2, random_state=42)

In [ ]:
user_train

In [ ]:
user_test

In [ ]:
ratings_df[ratings_df['userId'] == 2]

In [ ]:
movies_df

## Generowanie negatywów

In [ ]:
movies_per_user = ratings_df.groupby('userId')['movieId'].count()

In [ ]:
percentiles = movies_per_user.value_counts().sort_index().cumsum() / len(movies_per_user) * 100

# Plotting the percentiles
plt.figure(figsize=(10, 6))
plt.plot(percentiles.index, percentiles.values, marker='o')
plt.xlabel('Number of Movies Rated')
plt.ylabel('Percentile of Users (%)')
plt.title('Percentiles of Users by Number of Movies Watched')
plt.grid(True)
plt.xlim(0, 1000)
plt.show()

In [ ]:
ratings_df.groupby(['userId','movieId']).size().max()

In [ ]:
all_users_movie_combinations = pd.MultiIndex.from_product([ratings_df['userId'].unique(), movies_df['movieId'].unique()], names=['userId', 'movieId']).to_frame(index=False)

In [ ]:
all_users_movie_combinations.head()

In [ ]:
extended_ratings_df = pd.merge(all_users_movie_combinations, ratings_df, on=['userId', 'movieId'], how='left')
extended_ratings_df.fillna(0, inplace=True)

## Łączenie tabelek

In [ ]:
merged_df = pd.merge(movies_df, extended_ratings_df, on='movieId', how='inner')

In [ ]:
merged_df = merged_df.merge(users_df, on='userId')

In [ ]:
ratings_df['rating'].value_counts()

In [ ]:
merged_df['watched'] = (merged_df['rating'] > 0) * 1

In [ ]:
merged_df ['watched'].mean()

In [ ]:
merged_df

In [ ]:
merged_df['watched'].mean()

## Trening modelu do klasyfikacji binarnej

In [ ]:
merged_df

In [ ]:
merged_df_drop = merged_df.drop(columns=['rating', 'timestamp', 'title', 'movieId']).fillna(0)

In [ ]:
train = merged_df_drop[merged_df_drop['userId'].isin(user_train['userId'])]
test = merged_df_drop[merged_df_drop['userId'].isin(user_test['userId'])]

In [ ]:
positive_rows = train[train['watched'] == 1]
negative_rows = train[train['watched'] == 0]
sampled_negative_rows = negative_rows.groupby('userId', group_keys=False).sample(
    frac=0.05,
    random_state=42
)
reduced_train = pd.concat([positive_rows, sampled_negative_rows], ignore_index=True)
X_train = reduced_train.drop(columns=['watched', 'userId'])
y_train = reduced_train['watched']
X_test = test.drop(columns=['watched', 'userId'])
y_test = test['watched']

In [ ]:
X_train

In [ ]:
%%time

log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
log_reg.fit(X_train, y_train)

In [ ]:
y_test_proba = log_reg.predict_proba(X_test)[:, 1]

In [ ]:
y_test_proba[:10]

In [ ]:
test_results_df = merged_df.loc[test.index, ['userId', 'movieId']].copy()
test_results_df['watched'] = y_test.to_numpy()
test_results_df['probability'] = y_test_proba
test_results_df.head()

In [ ]:
user_id = test_results_df['userId'].iloc[0]
usr_tab = test_results_df[test_results_df['userId'] == user_id]

In [ ]:
usr_tab.sort_values('probability', ascending=False).head(10)

In [ ]:
usr_tab[usr_tab['watched'] == 1].sort_values('probability', ascending=False)

## Ocena rekomendacji

In [ ]:
user_ap_scores = test_results_df.groupby('userId')[['watched', 'probability']].apply(
    lambda group: average_precision_score(group['watched'], group['probability'])
)
model_mean_ap = user_ap_scores.mean()

random_results_df = test_results_df[['userId', 'watched']].copy()
random_results_df['probability'] = np.random.default_rng(42).random(len(random_results_df))
random_user_ap_scores = random_results_df.groupby('userId')[['watched', 'probability']].apply(
    lambda group: average_precision_score(group['watched'], group['probability'])
)

pd.Series({
    'model_mean_average_precision': model_mean_ap,
    'random_baseline_mean_average_precision': random_user_ap_scores.mean()
})